# PIDA-RL Diabetes — 15. Publicación temporal del dashboard con ngrok

Este cuaderno publica el dashboard V2 (`dashboard_streamlit/app.py`) desde Google Colab mediante un túnel de ngrok.

- **Fuente:** rama `main` del repositorio `JssSalas/pida-rl-datos-publicos`; los resultados agregados ya están en `dashboard_data/v2/`, así que no se necesita montar Drive ni volver a entrenar.
- **Secreto requerido:** `NGROK_AUTHTOKEN` en los secretos de Colab (icono de llave), con **Notebook access** activado. El valor nunca se escribe en el cuaderno ni se imprime.
- **Duración:** la URL es temporal. Deja de funcionar al cerrar el túnel, detener Streamlit o reiniciar el entorno. Para una URL permanente usa el cuaderno 16 (Streamlit Community Cloud).
- **Datos:** solo resultados agregados de simulación; no se publican microdatos de ENSANUT ni identificadores.

## 1. Inicialización

In [ ]:
# === Inicialización: clona o actualiza el repositorio en la rama main ===
from pathlib import Path
import importlib.util, os, subprocess, sys

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'main')
IN_COLAB = importlib.util.find_spec('google.colab') is not None

# En Colab el código vive en /content; fuera de Colab se usa PIDA_REPO_ROOT o la carpeta actual.
if IN_COLAB:
    REPO_ROOT = Path('/content/pida-rl-datos-publicos')
else:
    REPO_ROOT = Path(os.environ.get('PIDA_REPO_ROOT', Path.cwd() / 'pida-rl-datos-publicos')).resolve()

if (REPO_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '-q', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '-q', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, REPO_URL, str(REPO_ROOT)], check=True)

COMMIT = subprocess.run(['git', '-C', str(REPO_ROOT), 'rev-parse', '--short', 'HEAD'],
                        text=True, capture_output=True, check=True).stdout.strip()
APP_PATH = REPO_ROOT / 'dashboard_streamlit' / 'app.py'
DATA_DIR = REPO_ROOT / 'dashboard_data' / 'v2'
REQS = REPO_ROOT / 'dashboard_streamlit' / 'requirements.txt'
print(f'Colab: {IN_COLAB} | rama: {BRANCH} | commit: {COMMIT}')
print('Repositorio:', REPO_ROOT)


## 2. Dependencias del dashboard

In [ ]:
# Instala solo las dependencias del dashboard (las mismas que usará Streamlit Cloud).
pip = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQS)], text=True, capture_output=True)
if pip.returncode != 0:
    faltan = [m for m in ('streamlit', 'pandas', 'plotly') if importlib.util.find_spec(m) is None]
    if faltan:
        raise RuntimeError(f'pip falló y faltan {faltan}:\n{pip.stderr[-1500:]}')
    print('pip no disponible en este entorno; se usan las dependencias ya instaladas.')
import streamlit, pandas, plotly
print('streamlit', streamlit.__version__, '| pandas', pandas.__version__, '| plotly', plotly.__version__)


## 3. Verificación del repositorio y prueba de humo

In [ ]:
# Verifica que el repositorio tenga todo lo que necesita el dashboard y que no publique datos sensibles.
import csv, py_compile

REQUERIDOS = [
    'dashboard_streamlit/app.py', 'dashboard_streamlit/data_loader.py', 'dashboard_streamlit/requirements.txt',
    'dashboard_data/v2/manifest.csv', 'dashboard_data/v2/resumen_algoritmos.csv',
    'dashboard_data/v2/rendimiento_ic.csv', 'dashboard_data/v2/criterios_pida.csv',
]
faltan = [p for p in REQUERIDOS if not (REPO_ROOT / p).exists()]
assert not faltan, f'Faltan archivos en la rama {BRANCH}: {faltan}'
py_compile.compile(str(APP_PATH), doraise=True)

versionados = subprocess.run(['git', '-C', str(REPO_ROOT), 'ls-files'], text=True,
                             capture_output=True, check=True).stdout.splitlines()
PROHIBIDOS = ('.sav', '.dta', '.parquet', 'secrets.toml', '.env', 'cohorte_diabetes.csv')
sensibles = [f for f in versionados if f.endswith(PROHIBIDOS)]
assert not sensibles, f'Archivos que no deben publicarse: {sensibles}'

# Ningún CSV publicado debe traer identificadores de perfil o de vivienda.
COLUMNAS_PROHIBIDAS = {'FOLIO_INT', 'folio', 'perfil_id', 'id_perfil'}
for f in versionados:
    if f.startswith('dashboard_data/') and f.endswith('.csv'):
        with open(REPO_ROOT / f, newline='', encoding='utf-8') as fh:
            encabezado = set(next(csv.reader(fh)))
        assert not (encabezado & COLUMNAS_PROHIBIDAS), f'{f} contiene identificadores: {encabezado & COLUMNAS_PROHIBIDAS}'
print(f'Verificación correcta: {len(REQUERIDOS)} archivos requeridos, sin datos crudos ni identificadores.')


In [ ]:
# Prueba de humo: recorre todas las secciones de la barra lateral sin servidor ni túnel.
import warnings
from streamlit.testing.v1 import AppTest

os.environ['PIDA_DASHBOARD_DATA_DIR'] = str(DATA_DIR)
warnings.filterwarnings('ignore')
at = AppTest.from_file(str(APP_PATH), default_timeout=180).run()
secciones = list(at.sidebar.radio[0].options)
resultado = []
for seccion in secciones:
    at.sidebar.radio[0].set_value(seccion).run()
    errores = [e.value for e in at.exception] + [e.value for e in at.error]
    assert not errores, f'La sección «{seccion}» muestra errores: {errores[:2]}'
    resultado.append(f'{seccion}: {len(at.dataframe)} tablas')
print(f'Prueba de humo correcta en {len(secciones)} secciones, sin errores:')
print('\n'.join('  - ' + r for r in resultado))


## 4. Leer el token de ngrok

Se lee de los secretos de Colab o, fuera de Colab, de la variable de entorno `NGROK_AUTHTOKEN`. Solo se muestra si existe y su longitud, nunca su valor.

In [ ]:
def leer_secreto(nombre):
    if IN_COLAB:
        try:
            from google.colab import userdata
            valor = userdata.get(nombre)
            if valor:
                return valor.strip()
        except Exception as error:  # SecretNotFoundError o NotebookAccessError
            print(f'No se pudo leer {nombre} desde Colab: {type(error).__name__}')
    return (os.environ.get(nombre) or '').strip() or None

NGROK_AUTHTOKEN = leer_secreto('NGROK_AUTHTOKEN')
if NGROK_AUTHTOKEN:
    print(f'NGROK_AUTHTOKEN disponible ({len(NGROK_AUTHTOKEN)} caracteres).')
else:
    print('NGROK_AUTHTOKEN no disponible: agrega el secreto en Colab y activa Notebook access. '
          'La sección 5 iniciará Streamlit, pero sin túnel público.')


## 5. Iniciar Streamlit y abrir el túnel

In [ ]:
import socket, time

PORT = 8501

def puerto_abierto(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex(('127.0.0.1', port)) == 0

# Detiene una ejecución previa de este cuaderno, si existe.
if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate(); streamlit_process.wait(timeout=10)

env_app = dict(os.environ, PIDA_DASHBOARD_DATA_DIR=str(DATA_DIR))
LOG = open(REPO_ROOT / 'streamlit_ngrok.log', 'w')
streamlit_process = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', str(APP_PATH), '--server.port', str(PORT),
     '--server.headless', 'true', '--server.address', '127.0.0.1', '--browser.gatherUsageStats', 'false'],
    cwd=str(REPO_ROOT), env=env_app, stdout=LOG, stderr=subprocess.STDOUT)

for _ in range(60):
    if puerto_abierto(PORT) or streamlit_process.poll() is not None:
        break
    time.sleep(1)
assert puerto_abierto(PORT), 'Streamlit no inició; revisa streamlit_ngrok.log'
print(f'Streamlit activo en el puerto {PORT} (PID {streamlit_process.pid}).')

PUBLIC_URL = None
if NGROK_AUTHTOKEN:
    if importlib.util.find_spec('pyngrok') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok>=7.0'], check=True)
    from pyngrok import conf, ngrok
    conf.get_default().auth_token = NGROK_AUTHTOKEN
    for t in ngrok.get_tunnels():          # cierra túneles anteriores de esta sesión
        ngrok.disconnect(t.public_url)
    public_tunnel = ngrok.connect(PORT, 'http')
    PUBLIC_URL = public_tunnel.public_url
    print('Dashboard publicado temporalmente en:', PUBLIC_URL)
else:
    print('Sin túnel público. Streamlit quedó activo solo dentro del entorno.')


## 6. Comprobar la URL pública

ngrok puede mostrar una página de aviso la primera vez que alguien abre la URL desde el navegador; basta con pulsar **Visit Site**. Esta celda envía la cabecera que omite ese aviso para comprobar que la app responde.

In [ ]:
import urllib.request

objetivo = PUBLIC_URL or f'http://127.0.0.1:{PORT}'
req = urllib.request.Request(objetivo.rstrip('/') + '/_stcore/health',
                             headers={'ngrok-skip-browser-warning': '1'})
with urllib.request.urlopen(req, timeout=30) as r:
    print('Estado:', r.status, '| respuesta:', r.read().decode()[:20], '|', objetivo)


## 7. Cerrar la publicación temporal

Ejecuta esta celda al terminar la revisión.

In [ ]:
if PUBLIC_URL:
    from pyngrok import ngrok
    ngrok.disconnect(PUBLIC_URL); ngrok.kill()
    print('Túnel de ngrok cerrado.')
if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate(); streamlit_process.wait(timeout=10)
    print('Streamlit detenido.')
LOG.close()
